# Inside a digit-reversing Transformer

Run **Run → Run All Cells**, then scroll to **Explore the run**. Edit the PyTorch cells to change the model.

The encoder reads the digits in `HOLDOUT`. The decoder starts with `<bos>` and learns to emit the reversal followed by `<eos>`.

## Choose the architecture

Edit the settings below and run all cells again. Each attention head gets WIDTH / HEADS features. If you change LENGTH, give HOLDOUT that many digits.

In [ ]:
from copy import deepcopy
import math

import torch
from torch import nn
from torch.nn import functional as F

DIGITS = 10
LENGTH = 2
BOS = DIGITS       # Decoder input token marking the beginning of the answer.
EOS = DIGITS + 1   # Decoder token marking the end of the answer.
WIDTH = 32         # Features carried by each token between layers.
HEADS = 4          # Each head uses WIDTH / HEADS features.
INNER = 64         # Features inside the feed-forward layer.
BLOCKS = 1         # Number of blocks on each side.
BATCH = 64         # Examples per optimizer update.
STEPS = 800
RATE = 0.03       # Adam learning rate.
if not 2 <= DIGITS <= 10:
    raise ValueError("Use between 2 and 10 digit classes.")

# Keep the demo digits in range when changing the vocabulary size.
HOLDOUT = (3 % DIGITS, 8 % DIGITS)
SEED = 7

if len(HOLDOUT) != LENGTH or any(not 0 <= digit < DIGITS for digit in HOLDOUT):
    raise ValueError(f"HOLDOUT must contain {LENGTH} digits from 0 to {DIGITS - 1}.")

## 1. Attention

Start with <code>Attention.forward()</code>. A **query** asks what to read. A **key** determines how strongly that position matches the query. A **value** supplies the features to read. Each head has its own Q, K, and V weight rows. Open one projection in the diagram to see only that head’s rows.

A call to `show` returns the computed tensor and gives the viewer a name for that step. During replay, `show` also saves that result, its inputs, and its gradient for the diagram. During training, `passthrough` just returns the result.


In [ ]:
def passthrough(name, value, *parents, **info):
    """Ordinary training uses the tensor directly. A trace also records its inputs."""
    return value


class Attention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        # Each linear layer computes input @ weight.T + bias.
        self.query = nn.Linear(width, width)
        self.key = nn.Linear(width, width)
        self.value = nn.Linear(width, width)
        self.output = nn.Linear(width, width)
        self._heads = heads

    def forward(self, query_input, key_value_input, show, mask=None):
        # Self-attention takes both inputs from one stream.
        # Cross-attention takes queries from the decoder and keys/values from the encoder.
        # Compute all heads together. Each head owns a different set of output rows.
        queries = self.query(query_input).chunk(self._heads, dim=-1)
        keys = self.key(key_value_input).chunk(self._heads, dim=-1)
        values = self.value(key_value_input).chunk(self._heads, dim=-1)
        heads = []
        for head, (qh, kh, vh) in enumerate(zip(queries, keys, values), 1):
            # Draw each slice as a projection using only that head's rows of W and b.
            start = (head - 1) * qh.size(-1)
            qh = show(f"Query · head {head}", qh, query_input, op=self.query, output_start=start)
            kh = show(f"Key · head {head}", kh, key_value_input, op=self.key, output_start=start)
            vh = show(f"Value · head {head}", vh, key_value_input, op=self.value, output_start=start)

            # Compare every query to every key. The result has one row per query
            # and one column per key. For source 38 this is 3 × 2 in training.
            # Scaling keeps dot products from growing with the head width.
            raw_scores = qh @ kh.transpose(-2, -1)
            scores = show(f"Scores · head {head}", raw_scores / math.sqrt(qh.size(-1)),
                          qh, kh, kind="dot")

            # A masked score gets zero probability after softmax.
            if mask is not None:
                scores = show(f"Causal mask · head {head}", scores.masked_fill(mask, -torch.inf),
                              scores, kind="mask")

            # Each query row sums to one. Use it to form a weighted sum of value rows.
            weights = show(f"Softmax · head {head}", scores.softmax(-1),
                           scores, kind="softmax")
            heads.append(show(f"Weighted values · head {head}", weights @ vh,
                              weights, vh, kind="weighted"))

        # Join the head features and let the output projection mix them.
        joined = show("Concatenate heads", torch.cat(heads, -1), *heads, kind="join")
        return show("Attention output", self.output(joined), joined, op=self.output)

## 2. Encoder and decoder

Each block starts with self-attention. The decoder adds cross-attention. `encoder_output` is one final encoder vector per source digit. Decoder cross-attention projects those vectors into keys and values. The diagram shows those projections separately.

In [ ]:
def add_norm(x, change, norm, show):
    # Keep the input through the residual path, then normalize each token's features.
    x = show("Residual addition", x + change, x, change, kind="add")
    return show("Layer normalization", norm(x), x, op=norm, kind="norm")


class Block(nn.Module):
    def __init__(self, width, heads, inner, side):
        super().__init__()
        self.self_attention = Attention(width, heads)
        self.self_norm = nn.LayerNorm(width)
        if side == "decoder":
            self.cross_attention = Attention(width, heads)
            self.cross_norm = nn.LayerNorm(width)
        self.expand = nn.Linear(width, inner)
        self.contract = nn.Linear(inner, width)
        self.output_norm = nn.LayerNorm(width)

    def forward(self, x, show, block_name, encoder_output=None):
        def at(section):
            # Bind the diagram label once so each operation only needs a short name.
            def record(name, value, *parents, **info):
                return show(name, value, *parents, group=f"{block_name} · {section}", **info)

            return record

        # Row i can see columns 0 through i. Only decoder self-attention uses this mask.
        mask = None
        if encoder_output is not None:
            positions = x.size(1)
            mask = torch.ones(positions, positions, device=x.device, dtype=torch.bool).triu(1)

        # A self-attention layer gets queries and keys/values from the same tokens.
        record = at("Self-attention")
        change = self.self_attention(x, x, record, mask)
        x = add_norm(x, change, self.self_norm, record)
        if encoder_output is not None:
            # Every decoder position can read all source positions through cross-attention.
            record = at("Cross-attention")
            change = self.cross_attention(x, encoder_output, record)
            x = add_norm(x, change, self.cross_norm, record)

        # Apply the same two-layer network separately to each token.
        record = at("Feed-forward")
        wide = record("Expand", self.expand(x), x, op=self.expand)
        active = record("ReLU", wide.relu(), wide, kind="relu")
        narrow = record("Contract", self.contract(active), active, op=self.contract)
        return add_norm(x, narrow, self.output_norm, record)


class Transformer(nn.Module):
    def __init__(self, width=WIDTH, heads=HEADS, inner=INNER, blocks=BLOCKS):
        super().__init__()
        if heads < 1 or width % heads or width % 2 or blocks < 1:
            raise ValueError("Use an even width divisible by the head count and at least one block.")
        self.embedding = nn.Embedding(DIGITS + 2, width)
        self.encoder = nn.ModuleList(Block(width, heads, inner, "encoder") for _ in range(blocks))
        self.decoder = nn.ModuleList(Block(width, heads, inner, "decoder") for _ in range(blocks))
        # Predict ten digits or EOS. BOS is only an input.
        self.output = nn.Linear(width, DIGITS + 1)
        self._width = width

        # Give each position fixed sine/cosine features at multiple frequencies.
        # A buffer travels with model state but receives no optimizer updates.
        positions = torch.arange(LENGTH + 1)[:, None]
        freq = 10000 ** (-torch.arange(0, width, 2) / width)
        angles = positions * freq
        self.register_buffer("position", torch.stack((angles.sin(), angles.cos()), -1).flatten(1))

    def _embed(self, tokens, show, group):
        # Convert token IDs [batch, positions] to vectors [batch, positions, WIDTH].
        digit = show("Token embedding", self.embedding(tokens) * math.sqrt(self._width),
                     op=self.embedding, kind="embedding", group=group, tokens=tokens)
        position = show("Sinusoidal positions", self.position[:tokens.size(1)].expand_as(digit),
                        kind="position", group=group)
        return show("Embedding + position", digit + position, digit, position,
                    kind="add", group=group)

    def encode(self, source, show=passthrough):
        # Return one contextual vector per source digit.
        x = self._embed(source, show, "Encoder input")
        for i, block in enumerate(self.encoder, 1):
            x = block(x, show, f"Encoder {i}")
        return x

    def decode(self, prefix, encoder_output, show=passthrough):
        # Each prefix position produces scores for the next token.
        x = self._embed(prefix, show, "Decoder input")
        for i, block in enumerate(self.decoder, 1):
            x = block(x, show, f"Decoder {i}", encoder_output)
        return show("Next-token scores", self.output(x), x, op=self.output, group="Prediction")

    def forward(self, source, prefix, show=passthrough):
        return self.decode(prefix, self.encode(source, show), show)

## 3. Prepare examples and generate predictions

`practice` gives the decoder the correct prefix to teach it what comes next. `generate` starts at BOS and uses its own guesses. Compare those two functions to see why the same model can train on all positions at once and still generate one token at a time.

In [ ]:
def practice(model, source, show=passthrough):
    # Align inputs and targets one position apart so each row predicts the next token.
    # Source 02 gives inputs BOS 2 0 and targets 2 0 EOS.
    reverse = source.flip(1)
    prefix = torch.cat((torch.full_like(source[:, :1], BOS), reverse), 1)
    # Output classes 0 through DIGITS - 1 are digits. The final class is EOS.
    target = torch.cat((reverse, torch.full_like(source[:, :1], DIGITS)), 1)
    scores = model(source, prefix, show)
    # PyTorch expects [batch, classes, positions]. Keep one loss per target token.
    losses = F.cross_entropy(scores.transpose(1, 2), target, reduction="none")
    return prefix, target, losses


@torch.no_grad()
def generate(model, source):
    # Encode once. The decoder recomputes projected K and V on each pass.
    encoder_output = model.encode(source)
    prefix = torch.full_like(source[:, :1], BOS)
    predictions = []
    # Stop at EOS or at the maximum number of predictions this task allows.
    for _ in range(LENGTH + 1):
        # This function handles one example. The final prefix row predicts the next token.
        probs = model.decode(prefix, encoder_output)[0, -1].softmax(0)
        guess = probs.argmax().item()
        token = EOS if guess == DIGITS else guess
        predictions.append((token, probs))
        if token == EOS:
            break
        prefix = torch.cat((prefix, prefix.new_tensor([[token]])), 1)
    return predictions


def draw_batch():
    source = torch.randint(DIGITS, (BATCH, LENGTH))
    # If the demo input is drawn, change its first digit to keep it held out.
    source[(source == torch.tensor(HOLDOUT)).all(1), 0] = (HOLDOUT[0] + 1) % DIGITS
    return source

## 4. Train the model

One batch contains 64 examples. The loss averages all target positions, including EOS. Adam updates the weights once per batch.

In [ ]:
# Store the weights and optimizer immediately before each update.
torch.manual_seed(SEED)
torch.set_num_threads(2)
net = Transformer(WIDTH, HEADS, INNER, BLOCKS)
adam = torch.optim.Adam(net.parameters(), lr=RATE)
history = []
for step in range(STEPS):
    batch = draw_batch()
    # The viewer uses these copies to replay the exact batch and Adam update.
    snapshot = deepcopy(net.state_dict()), deepcopy(adam.state_dict()), batch

    # Predict the next token, then compute gradients and update the weights.
    _, _, losses = practice(net, batch)
    loss = losses.mean()
    adam.zero_grad()
    loss.backward()
    adam.step()
    history.append((*snapshot, loss.item()))

print(f"{STEPS} updates · batch loss {history[0][-1]:.5f} → {history[-1][-1]:.5f}")

## Replay and draw

These collapsed cells power the interactive figures. Expand them to see how Python records tensors and sends them to the browser.

In [ ]:
def numbers(t):
    # Convert tensors to JSON-friendly lists. The viewer displays the sentinel as −∞.
    return t.detach().nan_to_num(neginf=-1e30).tolist()


def replay(update, sample, digits):
    # Work on a private copy so replay and edits never modify the recorded run.
    params, optim, batch, _ = history[update - 1]
    net = Transformer(WIDTH, HEADS, INNER, BLOCKS)
    net.load_state_dict(params)
    adam = torch.optim.Adam(net.parameters(), lr=RATE)
    adam.load_state_dict(deepcopy(optim))
    batch = batch.clone()
    recorded = batch[sample].tolist()
    if digits:
        # BOS and EOS have IDs above the valid source digits.
        if len(digits) != LENGTH or any(not "0" <= digit <= str(DIGITS - 1) for digit in digits):
            raise ValueError(f"Enter {LENGTH} digits from 0 to {DIGITS - 1}.")
        batch[sample] = torch.tensor(list(map(int, digits)))

    # Build the diagram by mapping each tensor to the node that produced it.
    nodes, ids = [], {}
    names = {id(layer): name for name, layer in net.named_modules()}

    def show(name, value, *parents, group, kind="linear", op=None, **info):
        # Fixed position features also need a displayed gradient during replay.
        if not value.requires_grad:
            value = value.detach().requires_grad_()
        value.retain_grad()
        # Keep tensor identities so each diagram arrow can point to the correct parent.
        ids[id(value)] = len(nodes)
        if "tokens" in info:
            info["tokens"] = info["tokens"][sample].tolist()
        if kind == "norm":
            info["eps"] = op.eps
        meta = dict(name=name, group=group, kind=kind, layer=names.get(id(op)),
                    parents=[ids[id(parent)] for parent in parents], **info)
        nodes.append((meta, value))
        return value

    # Repeat the forward and backward pass, then export the selected example.
    prefix, target, losses = practice(net, batch, show)
    losses.mean().backward()
    source = batch[sample].tolist()
    count = losses.numel()
    result = dict(update=update, sample=sample+1, source=source, recorded=recorded,
                  tokens=prefix[sample].tolist(), target=target[sample].tolist(),
                  losses=numbers(losses[sample]), loss=losses.mean().item(),
                  count=count, width=WIDTH, eos=EOS, bos=BOS,
                  nodes=[dict(meta, v=numbers(v[sample]), g=numbers(v.grad[sample]))
                         for meta, v in nodes])
    group = adam.param_groups[0]
    result.update(rate=group["lr"], betas=list(group["betas"]), eps=group["eps"])
    # Generate before Adam steps so the predictions use the same weights as the diagram.
    result["guesses"] = [dict(guess=token, p=numbers(probs))
                         for token, probs in generate(net, batch[sample:sample+1])]

    # Save actual weight changes and the moment estimates that produced them.
    adam.step()
    result["params"] = {}
    for index, (name, p) in enumerate(net.named_parameters()):
        state = adam.state[p]
        prior = optim["state"].get(optim["param_groups"][0]["params"][index], {})
        result["params"][name] = dict(before=numbers(params[name]), after=numbers(p), g=numbers(p.grad),
            m=numbers(state["exp_avg"]), v=numbers(state["exp_avg_sq"]),
            m0=numbers(prior.get("exp_avg", torch.zeros_like(p))),
            v0=numbers(prior.get("exp_avg_sq", torch.zeros_like(p))))
    return result

In [ ]:
diagram_js = r"""
function render({model, el}) {
  const root = el.attachShadow({mode:"open"});
  root.innerHTML = '<style>'+model.get("css")+`
.viewer-bar{display:flex;align-items:center;justify-content:space-between;gap:12px;flex-wrap:wrap;position:sticky;top:0;z-index:2;background:white;border-bottom:1px solid #ccc;padding:10px 0}
.viewer-bar input{width:clamp(100px,24vw,235px)}.viewer-bar output{min-width:4ch;font-variant-numeric:tabular-nums}
.direction{display:inline-flex;gap:4px}.direction button[aria-pressed=true]{background:#00695c;color:white;outline:none}
.map-layout{display:grid;grid-template-columns:minmax(0, .84fr) minmax(0,1.16fr);gap:16px;align-items:start}
.diagram-panel{min-width:0}.diagram-panel h3,.scalar-panel h3{margin:0 0 8px}
.diagram{overflow:auto}.diagram svg{display:block;min-width:290px;width:100%}
.diagram [role=button]{cursor:pointer}.diagram [role=button]:hover rect,.diagram [role=button]:focus-visible rect{stroke:#00695c;stroke-width:2.5}
.diagram text{pointer-events:none}.diagram .node-title{font-size:12px}.diagram .node-subtitle{font-size:10px;fill:#5f6368}
.diagram .section-label{fill:#00695c;font-size:12px}.diagram .edge{pointer-events:none}
.scalar-panel{margin-top:20px;border-top:1px solid #ccc;padding-top:16px}
#grid{width:max-content;max-width:100%;box-sizing:border-box}
#grid button{min-width:53px;height:31px;white-space:nowrap;font:11px ui-monospace,monospace;padding:0 3px}
#grid .idx{font-size:12px;align-self:center;padding:0 5px}.scalar-panel p{margin:.6em 0}
#math details,#weights details,#loss-plot,#generated,#example-setup{margin:.65em 0}
#math .equation,#weights .equation{margin:.65em 0}
.result{font-variant-numeric:tabular-nums;font-weight:500}
.operand-graph{max-width:680px}.operand-graph button{border-radius:4px}
.numeric-table{overflow:auto}.numeric-table table{border-collapse:collapse;font-variant-numeric:tabular-nums}
.numeric-table td,.numeric-table th{padding:4px 8px;text-align:right;border-bottom:1px solid #ccc}
.numeric-table textarea{width:96%;min-height:7em}.numeric-table button{font-family:ui-monospace,monospace;padding:1px 4px;border-radius:4px}
.projection-weights{display:flex;flex-wrap:wrap;gap:16px}.projection-weights section{max-width:100%}
.equation{overflow-wrap:anywhere;font-family:monospace}
#prefix,#status,.graph-help{font-size:14px}
@media(max-width:680px){.map-layout{grid-template-columns:1fr}.viewer-bar{position:static}}

</style><main></main>`;
  const main = root.querySelector("main");
  main.innerHTML = `
  <h1>Inside a digit-reversing Transformer</h1>
  <p class="post-summary">Open a module, choose an operation, then click one value to see the arithmetic.</p>
  <div class="viewer-bar">
    <label>Training update <input id="step" type="range" min="1" value="1"><output id="stepno">1</output></label>
    <div class="direction" role="group" aria-label="Pass direction">
      <button id="forward" aria-pressed="true">Forward</button><button id="backward" aria-pressed="false">Backward</button>
    </div>
  </div>
  <p id="prefix"></p>
  <div class="map-layout">
    <section class="diagram-panel"><h3>Model</h3><div id="graph" class="diagram"></div></section>
    <section class="diagram-panel" id="module-panel" hidden>
      <h3 id="module-title"></h3><div id="operations" class="diagram"></div>
    </section>
  </div>
  <section class="scalar-panel" id="inspector" hidden>
    <h3 id="title"></h3><p id="why"></p><p id="axes"></p>
    <div id="grid" role="group" aria-label="Token and feature values"></div>
    <p id="cell" class="result"></p><div id="scalar" class="diagram operand-graph"></div>
    <div id="math"></div><div id="weights"></div>
  </section>
  <details id="loss-plot"><summary>Batch loss</summary>
    <svg id="curve" viewBox="0 0 600 95" aria-label="Recorded batch loss by update"></svg>
  </details>
  <details id="generated"><summary>Generated answer</summary>
    <div id="sequence"></div><p id="guess-detail"></p><div id="odds"></div>
  </details>
  <details id="example-setup"><summary>Change the example</summary>
    <div class="controls"><label>Example in batch <input id="sample" type="number" min="1" value="1"></label>
    <label>Digits <input id="custom" placeholder="recorded" inputmode="numeric" size="7"></label>
    <button id="apply">Try digits</button><button id="reset">Use recorded example</button></div>
    <p id="status" role="status"></p>
  </details>
`;
  const $ = id => root.getElementById(id);
  const history = model.get("history");
  const length = model.get("length");
  $("custom").maxLength=length;
  const f = x => x < -1e29 ? "−∞" : Math.abs(x)>0 && Math.abs(x)<0.0001 ? x.toExponential(4) : Number(x.toFixed(5)).toString();
  
  const fmt = arr => arr.map(f).join(" + ");
  const psoft = z => {const m=Math.max(...z),e=z.map(x=>Math.exp(x-m)),s=e.reduce((a,b)=>a+b);return e.map(x=>x/s)};
  const detail = (label, terms) => '<details><summary>'+label+'</summary><p class="numbers">'+terms+'</p></details>';
  let d, layer=null,row=0,col=0,back=false,out=0,parameterCell=null;
  let request=0, inFlight=null, queued=null;
  const plot=$("curve");
  const plotLeft=35, plotWidth=550, lastIndex=Math.max(1,history.length-1);
  const maxLoss=Math.max(...history);
  const x=i=>plotLeft+i*plotWidth/lastIndex;
  const y=loss=>72-58*loss/maxLoss;
  $("step").max=history.length;
  $("sample").max=model.get("batch");

  // The chart stays mounted during replay so dragging keeps pointer capture.
  plot.innerHTML='<text x="5" y="12">Recorded batch loss</text><path stroke="#ccc" fill="none" d="M35 14 V72 H585"/>'+
    '<polyline fill="none" stroke="#00695C" stroke-width="2" points="'+history.map((v,i)=>x(i)+','+y(v)).join(' ')+'"/>'+
    '<circle r="5" fill="#8e068e"><title></title></circle><text x="35" y="91">1</text>'+
    '<text x="470" y="91">Training update → '+history.length+'</text>';
  const marker=plot.querySelector("circle");
  function mark(update){
    const loss=history[update-1];
    marker.setAttribute("cx",x(update-1));
    marker.setAttribute("cy",y(loss));
    marker.querySelector("title").textContent="Update "+update+" · loss "+f(loss);
    $("stepno").textContent=update;
  }

  // Keep one Python replay in flight. Replace queued positions as the pointer moves.
  function sendReplay(){
    if(inFlight || !queued){return;}
    inFlight={...queued,id:++request};
    queued=null;
    model.set("selection",inFlight);
    model.save_changes();
  }
  function refresh(){
    const digits=$("custom").value;
    const maxDigit=d.bos-1;
    if(digits && (digits.length!==length || [...digits].some(digit=>digit<"0" || digit>String(maxDigit)))){
      $("status").textContent="Enter exactly "+length+" digits from 0 to "+maxDigit+".";
      return;
    }
    const sample=+$("sample").value;
    if(!Number.isInteger(sample)||sample<1||sample>model.get("batch")){ $("status").textContent="Choose an example inside the batch."; return; }
    queued={update:+$("step").value,sample,digits};
    sendReplay();
  }
  function scrub(update){
    $("step").value=update;
    mark(update);
    refresh();
  }
  // The loss plot feeds the same slider and replay queue.
  function bindScrubber(svg,left,width,intervals){
    svg.style.touchAction="none";
    svg.style.cursor="ew-resize";
    function stepAt(event){
      const point=new DOMPoint(event.clientX,event.clientY).matrixTransform(svg.getScreenCTM().inverse());
      const index=Math.round((point.x-left)*intervals/width);
      return 1+Math.min(history.length-1,Math.max(0,index));
    }
    svg.onpointerdown=event=>{
      if(event.button!==0){return;}
      svg.setPointerCapture(event.pointerId);
      scrub(stepAt(event));
    };
    svg.onpointermove=event=>{
      if(!svg.hasPointerCapture(event.pointerId)){return;}
      const update=stepAt(event);
      if(update!==+$("step").value){scrub(update);}
    };
  }
  bindScrubber(plot,plotLeft,plotWidth,lastIndex);
  $("step").oninput=()=>scrub(+$("step").value);
  $("sample").oninput=$("apply").onclick=refresh;
  $("reset").onclick=()=>{$("custom").value="";refresh()};
  $("custom").onkeydown=e=>{if(e.key==="Enter"){refresh()}};
  const token=x=>x===d.bos?"&lt;bos&gt;":x===d.eos?"&lt;eos&gt;":x;
  // Pick an output position and show the next-token probabilities from that pass.
  function generation(){
    out=Math.min(out,d.guesses.length-1);
    const expected=[...d.source.slice().reverse(),d.eos].map(token).join(' ');
    const buttons=d.guesses.map((g,i)=>'<button aria-pressed="'+(i===out)+'" data-i="'+i+
      '" title="Inspect guess '+(i+1)+'">'+token(g.guess)+'</button>').join(' ');
    const stop=d.guesses.at(-1).guess===d.eos?'Stopped at &lt;eos&gt;.':
      'Stopped at the '+d.guesses.length+'-token limit without predicting &lt;eos&gt;.';
    const checkpoint=d.update===1?'0 updates · untrained':(d.update-1)+' updates completed';
    $("sequence").innerHTML='<b>Model guesses</b> ('+checkpoint+') '+buttons+
      '<br><b>Expected</b> '+expected+'<br>'+stop;
    $("sequence").querySelectorAll("button").forEach(b=>b.onclick=()=>{out=+b.dataset.i;generation()});

    // Each guess is fed back to the decoder and changes the next prediction.
    const prefix=[d.bos,...d.guesses.slice(0,out).map(g=>g.guess)].map(token).join(' ');
    const guess=d.guesses[out];
    const probability=guess.p[guess.guess===d.eos?d.bos:guess.guess];
    $("guess-detail").innerHTML='Encoder reads <b>'+d.source.join(' ')+'</b>. Decoder reads <b>'+
      prefix+'</b> and picks <b>'+token(guess.guess)+'</b> because it has the highest probability ('+
      (100*probability).toFixed(2)+'%).';
    $("odds").innerHTML=d.guesses[out].p.map((v,i)=>'<div title="Token '+token(i===d.bos?d.eos:i)+' · '+f(v)+'"><b>'+token(i===d.bos?d.eos:i)+'</b><i style="width:'+100*v+'%"></i><small>'+f(100*v)+'%</small></div>').join('');
  }
  const notes={
    embedding:"Look up a learned row for the token and scale it before adding position information.",
    position:"Sine and cosine at different frequencies give the model a fixed signal for each position.",
    add:"Addition keeps both streams in the output. On the way back, both receive the same incoming gradient.",
    linear:"One output feature is the dot product of the input and a learned weight row, plus bias.",
    slice:"Each head reads a different contiguous slice of the projected features.",
    dot:"Compare one query with one key. Scale by the square root of the head width.",
    mask:"Future decoder positions become negative infinity, which gives them zero probability.",
    softmax:"Softmax turns the scores in one row into weights. Subtract the row maximum, exponentiate, then divide by the sum of the exponentials.",
    weighted:"For one output feature, multiply each value by its attention probability and sum.",
    join:"Place the features from the heads next to one another. The output projection mixes them.",
    norm:"Normalize each token across its features, then apply a learned scale and shift.",
    relu:"Keep positive values and replace negative values with zero.",
    loss:"Cross-entropy penalizes small probability assigned to the expected token, including EOS."
  };
  // i: query or token row, j: key row, k: feature, c: output coordinate.
  const forwardFormula={
    embedding:"x[i,c] = √d_model · E[token[i],c]",
    position:"PE[i,2k] = sin(i / 10000^(2k/d_model)); PE[i,2k+1] = cos(i / 10000^(2k/d_model))",
    add:"y[i,c] = a[i,c] + b[i,c]",
    linear:"Y = XWᵀ + b; y[i,c] = Σ_k X[i,k] · Wᵀ[k,c] + b[c]",
    slice:"head[i,c] = projection[i,head_offset+c]",
    dot:"S[i,j] = (Σ_k Q[i,k] · K[j,k]) / √d_head",
    mask:"M[i,j] = S[i,j] if j ≤ i, otherwise −∞",
    softmax:"p[i,c] = exp(s[i,c] − m) / Σ_j exp(s[i,j] − m), where m = max_j s[i,j]",
    weighted:"C[i,c] = Σ_j A[i,j] · V[j,c]",
    join:"joined[i,head_offset+c] = head[i,c]",
    norm:"μ = Σ_k x[k]/d; σ² = Σ_k (x[k]−μ)²/d; y[c] = γ[c] · (x[c]−μ)/√(σ²+ε) + β[c]",
    relu:"y[i,c] = max(0, x[i,c])",
    loss:"ℓ[i] = −ln p[i,target[i]]; L = (Σ ℓ)/N across the batch",
  };
  const equation=text=>'<p class="equation"><code>'+text+'</code></p>';
  const matrixInitial=model.get("initial_tensors");
  function matrixTable(values, labels, parameter=null, rowOffset=0){
    const vector=!Array.isArray(values[0]);
    const rows=vector?[values]:values;
    const fmtCell=v=>v < -1e29?"−∞":Number(v.toPrecision(8)).toString();
    const headings=rows[0].map((_,i)=>'<th scope="col">'+i+'</th>').join('');
    const body=rows.map((r,i)=>'<tr><th scope="row">'+(vector?"value":labels?.[i]??i)+
      '</th>'+r.map((v,j)=>'<td>'+(parameter===null?fmtCell(v):
        '<button data-weight="'+parameter+'" data-wrow="'+(rowOffset+(vector?j:i))+'" data-wcol="'+(vector?0:j)+'">'+
        fmtCell(v)+'</button>')+'</td>').join('')+'</tr>').join('');
    const tsv=rows.map(r=>r.map(v=>v < -1e29?"-Infinity":v).join("\t")).join("\n");
    return '<div class="numeric-table"><table><thead><tr><th></th>'+headings+
      '</tr></thead><tbody>'+body+'</tbody></table><details><summary>Full precision · copy values</summary>'+
      '<textarea readonly aria-label="Full precision values">'+tsv+'</textarea></details></div>';
  }

  const sum=xs=>xs.reduce((a,b)=>a+b,0);
  function normalize(xs,eps){
    const mean=sum(xs)/xs.length, centered=xs.map(x=>x-mean);
    const variance=sum(centered.map(x=>x*x))/xs.length;
    const sd=Math.sqrt(variance+eps);
    return {mean,variance,sd,values:centered.map(x=>x/sd)};
  }
  const escape=s=>String(s).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;");
  let activeGroup=null;
  const colors={teal:"#00695c", purple:"#8e068e", edge:"#a9b6b4", node:"#e8f3f0"};
  const dimensions={width:124,height:36,stage:61,gutter:18};
  function drawGraph(target,nodes,edges,width,height,activate,lanes=[]){
    const nodeMap=new Map(nodes.map(n=>[n.id,n]));
    const arrow=target.id+"-arrow";
    const stroke=back?colors.purple:colors.teal;
    const paths=edges.map(([from,to])=>{
      const a=nodeMap.get(back?to:from),b=nodeMap.get(back?from:to);
      if(!a||!b||a===b){return "";}
      const down=b.y>a.y, sameLevel=a.y===b.y;
      const startY=a.y+(down?1:-1)*a.h/2, endY=b.y+(down?-1:1)*b.h/2;
      const middle=(startY+endY)/2;
      const edge= sameLevel
        ? "M"+(a.x+Math.sign(b.x-a.x)*a.w/2)+" "+a.y+" H"+(b.x-Math.sign(b.x-a.x)*b.w/2)
        : "M"+a.x+" "+startY+" C"+a.x+" "+middle+" "+b.x+" "+middle+" "+b.x+" "+endY;
      const highlighted=a.selected||b.selected;
      return '<path class="edge" d="'+edge+'" stroke="'+(highlighted?stroke:colors.edge)+
        '" stroke-width="'+(highlighted?2:1.2)+'" marker-end="url(#'+arrow+')" fill="none"><title>'+
        escape(a.label+" → "+b.label)+'</title></path>';
    }).join('');
    const boxes=nodes.map((n,i)=>{
      const fill=n.selected?"#fde100":n.external?"#f5f5f5":colors.node;
      const interactive=n.action!==undefined;
      return '<g '+(interactive?'role="button" tabindex="0" data-index="'+i+'" aria-label="'+escape(n.accessible??n.label)+
        '" aria-pressed="'+!!n.selected+'"':"")+'><title>'+escape(n.tooltip??n.label)+'</title>'+
        '<rect x="'+(n.x-n.w/2)+'" y="'+(n.y-n.h/2)+'" width="'+n.w+'" height="'+n.h+
        '" rx="5" fill="'+fill+'" stroke="'+(n.selected?stroke:"#9bb3ad")+'"'+(n.external?' stroke-dasharray="3 2"':'')+'/>'+
        '<text class="node-title" text-anchor="middle" x="'+n.x+'" y="'+(n.y+(n.sub?-3:4))+'">'+escape(n.label)+'</text>'+
        (n.sub?'<text class="node-subtitle" text-anchor="middle" x="'+n.x+'" y="'+(n.y+11)+'">'+escape(n.sub)+'</text>':'')+'</g>';
    }).join('');
    const labels=lanes.map(([x,y,label])=>'<text class="section-label" text-anchor="middle" x="'+x+'" y="'+y+'">'+escape(label)+'</text>').join('');
    target.innerHTML='<svg viewBox="0 0 '+width+' '+height+'" role="group"><defs><marker id="'+arrow+
      '" markerWidth="6" markerHeight="6" refX="5" refY="3" orient="auto"><path d="M0 0 L6 3 L0 6 Z" fill="'+stroke+
      '"/></marker></defs>'+labels+paths+boxes+'</svg>';
    target.querySelectorAll('[data-index]').forEach(button=>{
      const act=()=>activate(nodes[+button.dataset.index].action);
      button.onclick=act;
      button.onkeydown=e=>{
        if(e.key==="Enter"||e.key===" "){e.preventDefault();act();}
      };
    });
  }

  function overview(){
    const groups=[...new Set(d.nodes.map(n=>n.group))];
    const nodes=[], edges=new Set();
    const groupLabel=g=>g.replace(/^(Encoder|Decoder) \d+ · /,"").replace(/^(Encoder|Decoder) input$/,"Input");
    const laneX={Encoder:82,Decoder:252};
    for(const side of Object.keys(laneX)){
      const lane=groups.filter(g=>g.startsWith(side));
      lane.forEach((g,i)=>{
        nodes.push({id:g,action:g,label:groupLabel(g),
          sub:i?g.split(" · ")[0]:(side==="Encoder"?d.source:d.tokens).map(tokenText).join(" "),
          selected:activeGroup===g,x:laneX[side],y:44+i*53,w:136,h:36});
      });
    }
    const predictionY=Math.max(...nodes.map(n=>n.y))+53;
    nodes.push({id:"Prediction",action:"Prediction",label:"Scores and loss",selected:activeGroup==="Prediction",
      x:laneX.Decoder,y:predictionY,w:136,h:36});
    for(const n of d.nodes){
      for(const parent of n.parents){
        const group=d.nodes[parent].group;
        if(group!==n.group){edges.add(group+"|"+n.group);}
      }
    }
    drawGraph($("graph"),nodes,[...edges].map(e=>e.split("|")),334,predictionY+26,openGroup,
      [[laneX.Encoder,15,"Encoder"],[laneX.Decoder,15,"Decoder"]]);
  }

  function openGroup(group){
    activeGroup=group;
    layer=null;
    row=col=0;
    repaint();
    $("operations").querySelector('[role=button]')?.focus({preventScroll:true});
  }

  function operationMap(){
    if(activeGroup===null){
      $("module-panel").hidden=true;
      return;
    }
    $("module-panel").hidden=false;
    $("module-title").textContent=activeGroup;
    const local=d.nodes.flatMap((n,i)=>n.group===activeGroup?[i]:[]);
    const localSet=new Set(local);
    const external=[...new Set(local.flatMap(i=>d.nodes[i].parents.filter(p=>!localSet.has(p))))];
    const levels=new Map(external.map(i=>[i,0]));
    const rows=external.length?[external]:[];
    const edges=[];
    for(const i of local){
      const parents=[...new Set(d.nodes[i].parents)];
      const level=parents.length?1+Math.max(...parents.map(p=>levels.get(p))):0;
      levels.set(i,level);
      (rows[level]??=[]).push(i);
      edges.push(...parents.map(p=>[p,i]));
    }
    const {width,height,stage,gutter}=dimensions;
    const columnWidth=width+gutter;
    const headFor=i=>+(d.nodes[i].name.match(/ · head (\d+)$/)?.[1]??0);
    const headCount=Math.max(0,...local.map(headFor));
    const maxPerHead=Math.max(1,...rows.flatMap(ids=>{
      const counts=new Map();
      for(const i of ids){
        const head=headFor(i);
        if(head){counts.set(head,(counts.get(head)??0)+1);}
      }
      return [...counts.values()];
    }));
    const headWidth=maxPerHead*columnWidth;
    const canvasWidth=Math.max(300,headCount*headWidth+gutter,
      Math.max(...rows.map(ids=>ids.length))*columnWidth+gutter);
    const nodes=rows.flatMap((ids,level)=>ids.map((i,index)=>{
      const n=d.nodes[i],foreign=!localSet.has(i);
      const parts=n.name.split(" · "), head=foreign?0:headFor(i);
      const siblings=head?ids.filter(j=>headFor(j)===head):ids;
      const laneCenter=head?gutter/2+(head-.5)*headWidth:canvasWidth/2;
      const x=laneCenter+(siblings.indexOf(i)-(siblings.length-1)/2)*columnWidth;
      return {id:i,action:i,label:parts[0],sub:foreign?n.group.split(" · ").at(-1):parts[1],
        tooltip:n.group+" · "+n.name,accessible:foreign?n.group+" · "+n.name:n.name,
        external:foreign,selected:layer===i,x,y:24+level*stage,w:width,h:height};
    }));
    drawGraph($("operations"),nodes,edges,canvasWidth,rows.length*stage,select);
    // Keep dense heads legible on narrow panes by allowing horizontal scroll.
    $("operations").querySelector("svg").style.minWidth=Math.min(canvasWidth,720)+"px";
  }
  function tokenText(value){return value===d.bos?"<bos>":value===d.eos?"<eos>":String(value);}
  function repaint(){
    overview();operationMap();
    $("inspector").hidden=layer===null;
    if(layer!==null){matrix();}
  }
  function select(i,r=row,c=col){
    layer=i;
    activeGroup=d.nodes[i].group;
    row=r;col=c;
    parameterCell=null;
    repaint();
  }
  function direction(reverse){
    back=reverse;
    $("forward").setAttribute("aria-pressed",!reverse);
    $("backward").setAttribute("aria-pressed",reverse);
    repaint();
  }
  $("forward").onclick=()=>direction(false);
  $("backward").onclick=()=>direction(true);

  // Expand the forward formula for one selected tensor cell.
  function calc(n,r,c){
    const [a,b]=n.parents.map(i=>d.nodes[i].v);
    const p=d.params[n.layer+".weight"], bias=d.params[n.layer+".bias"]?.before;
    const terms=xs=>detail("Show all "+xs.length+" terms",xs.join(" + "));
    const weightRow=(n.output_start??0)+c;
    switch(n.kind){
      case "embedding":return 'embedding['+token(n.tokens[r])+', '+c+'] × √'+d.width+' = '+f(p.before[n.tokens[r]][c])+' × √'+d.width;
      case "position":{
        const power=2*Math.floor(c/2)/d.width, angle=r/10000**power;
        return (c%2?'cos':'sin')+'('+r+' / 10000^'+f(power)+') = '+(c%2?'cos':'sin')+'('+f(angle)+')';
      }
      case "add":return f(a[r][c])+' + '+f(b[r][c]);
      case "linear":{
        const inputs=a[r].length, outputs=n.v[r].length;
        return equation("("+a.length+" × "+inputs+") · ("+inputs+" × "+outputs+") → ("+
          a.length+" × "+outputs+")")+
          'PyTorch stores W as '+outputs+' × '+inputs+'. Transposing swaps the indices, so Wᵀ[k,c] = W[c,k]. '+
          'Take the dot product with column '+c+' and add bias '+f(bias[weightRow])+'.'+
          terms(a[r].map((x,j)=>f(x)+' × '+f(p.before[weightRow][j])));
      }
      case "slice":return 'Copy projected feature '+(n.start+c)+' at row '+r+'.';
      case "dot":return 'Sum the products and divide by √'+a[r].length+'.'+terms(a[r].map((x,j)=>f(x)+' × '+f(b[c][j])));
      case "mask":return c>r?'The key is in the future. Replace the score with −∞.':'The key is visible. Keep '+f(a[r][c])+'.';
      case "softmax":{
        const max=Math.max(...a[r]), exps=a[r].map(x=>Math.exp(x-max));
        return 'For this row, max = '+f(max)+'. The selected weight is exp('+f(a[r][c])+' − ('+f(max)+
          ')) / '+f(sum(exps))+' = '+f(n.v[r][c])+'. Subtracting the same max keeps the ratios and prevents overflow.'+
          terms(exps.map(f));
      }
      case "weighted":return 'Sum attention × value across keys.'+terms(a[r].map((x,j)=>f(x)+' × '+f(b[j][c])));
      case "join":{
        let start=0;
        for(const [i,parent] of n.parents.entries()){
          const v=d.nodes[parent].v;
          if(c<start+v[r].length){return 'Copy head '+(i+1)+', feature '+(c-start)+'.';}
          start+=v[r].length;
        }break;
      }
      case "norm":{
        const {mean,variance,sd,values}=normalize(a[r],n.eps);
        return 'Across this row, μ = '+f(mean)+', variance = '+f(variance)+
          '. Divide ('+f(a[r][c])+' − μ) by √(variance + '+f(n.eps)+') = '+f(sd)+
          '. Then '+f(values[c])+' × '+f(p.before[c])+' + '+f(bias[c])+'.'+
          detail("Show the row used for mean and variance",a[r].map(f).join(", "));
      }
      case "relu":return 'max(0, '+f(a[r][c])+')';
      case "loss":return 'Expected token '+token(d.target[r]===d.bos?d.eos:d.target[r])+'. Loss = −ln('+f(a[r][d.target[r]])+').';
    }
    return '';
  }
  // Use the chain rule and add contributions from every operation that reads this cell.
  function backward(n,r,c){
    $("scalar").replaceChildren();
    if(n.kind==="loss"){return equation("∂L/∂ℓ[i] = 1/N")+'The batch mean sends 1 / '+d.count+' to each token loss.';}
    const pieces=[], contributions=[];
    d.nodes.forEach((child,id)=>child.parents.forEach((p,side)=>{
      if(p!==layer){return;}
      const [a,b]=child.parents.map(i=>d.nodes[i].v),g=child.g;
      let terms=[],rule,formula;
      switch(child.kind){
        case "linear":
          formula="g_x[i,c] = Σ_k g_y[i,k] · W[k,c]";rule="Multiply each downstream gradient by the corresponding projection weight";
          terms=g[r].map((x,j)=>x*d.params[child.layer+".weight"].before[(child.output_start??0)+j][c]);break;
        case "add":formula="g_a[i,c] = g_b[i,c] = g_y[i,c]";rule="Copy the incoming gradient";terms=[g[r][c]];break;
        case "relu":formula="g_x[i,c] = g_y[i,c] · 1(x[i,c] &gt; 0)";rule="Pass only positive inputs";terms=[n.v[r][c]>0?g[r][c]:0];break;
        case "mask":formula="g_S[i,j] = g_M[i,j] · 1(j ≤ i)";rule="Drop gradients at masked positions";terms=[c<=r?g[r][c]:0];break;
        case "slice":formula="g_projection[i,offset+c] receives g_head[i,c]";rule="Send the gradient to the copied features";terms=[g[r][c-child.start]??0];break;
        case "join":{
          formula="g_head[i,c] = g_joined[i,offset+c]";rule="Return the gradient to the head supplying the feature";
          const offset=sum(child.parents.slice(0,side).map(i=>d.nodes[i].v[r].length));
          terms=[g[r][offset+c]];break;
        }
        case "dot":{
          formula=side===0?"g_Q[i,c] = Σ_j g_S[i,j] · K[j,c] / √d_head":
            "g_K[j,c] = Σ_i g_S[i,j] · Q[i,c] / √d_head";
          rule="Multiply score gradients by the other vector and divide by √head width";
          const scale=Math.sqrt(a[0].length);
          terms=side===0?g[r].map((x,j)=>x*b[j][c]/scale):g.map((v,j)=>v[r]*a[j][c]/scale);break;
        }
        case "softmax":{
          formula="g_s[i,c] = Σ_j g_p[i,j] · p[i,j] · (1(j=c) − p[i,c])";rule="For every column j use upstream[j] × p[j] × (indicator[j=c] − p[c])";
          const p=child.v[r];terms=p.map((x,j)=>g[r][j]*x*((j===c?1:0)-p[c]));break;
        }
        case "weighted":
          formula=side===0?"g_A[i,j] = Σ_c g_C[i,c] · V[j,c]":"g_V[j,c] = Σ_i g_C[i,c] · A[i,j]";
          rule="Differentiate the weighted sum";
          terms=side===0?g[r].map((x,j)=>x*b[c][j]):g.map((v,j)=>v[c]*a[j][r]);break;
        case "norm":{
          formula="h = g_y · γ; z = (x−μ)/√(σ²+ε); g_x[c] = (h[c] − mean(h) − z[c]·mean(h·z))/√(σ²+ε)";rule="Subtract the mean of the scaled gradients and their projection onto the normalized row, then divide by standard deviation";
          const {sd,values}=normalize(a[r],child.eps);
          const h=g[r].map((x,j)=>x*d.params[child.layer+".weight"].before[j]);
          terms=[h[c]/sd,-sum(h)/h.length/sd,-values[c]*sum(h.map((x,j)=>x*values[j]))/h.length/sd];break;
        }
        case "loss":formula="g_p[i,c] = −1(c=target[i]) / (N · p[i,c])";rule="Only the expected token contributes −1 / (batch token count × probability)";
          terms=[c===d.target[r]?-1/(d.count*n.v[r][c]):0];break;
      }
      if(terms.length){
        contributions.push({id:contributions.length,node:id,label:child.name.split(" · ")[0],value:sum(terms)});
        pieces.push('<p><button data-node="'+id+'">'+child.name+'</button> '+rule+' = <b>'+f(sum(terms))+
        '</b></p>'+equation(formula)+(terms.length>1?detail("Show the contributions",fmt(terms)):""));}
    }));
    if(contributions.length){
      const stride=140,width=Math.max(280,stride*contributions.length);
      const nodes=contributions.map(({id,node,label,value},i)=>({id,action:node,label,sub:f(value),
        x:(width-(contributions.length-1)*stride)/2+i*stride,y:85,w:128,h:36}));
      nodes.push({id:"gradient",label:"Sum contributions",sub:f(n.g[r][c]),selected:true,x:width/2,y:22,w:140,h:36});
      drawGraph($("scalar"),nodes,contributions.map(x=>["gradient",x.id]),width,110,select);
      $("scalar").querySelector("svg").style.minWidth=Math.min(width,720)+"px";
    }
    return equation("∂L/∂x = Σ_y (∂L/∂y) · (∂y/∂x)")+
      'The chain rule adds each downstream contribution. g_x is shorthand for ∂L/∂x.'+pieces.join('');
  }
  function math(){
    const n=d.nodes[layer], value=(back?n.g:n.v)[row][col];
    const inspector=$("math");
    const open=new Set([...inspector.querySelectorAll('details[open]')].map(el=>el.firstElementChild.textContent));
    $("cell").textContent=(back?"∂ batch loss / ∂ cell":"Cell")+" ["+row+", "+col+"] = "+f(value);
    if(!back){scalarGraph(n,row,col);}
    const inputTables=n.parents.map((id,i)=>{
      const parent=d.nodes[id];
      return '<details><summary>Input '+(i+1)+' · '+escape(parent.name)+'</summary>'+
        matrixTable(parent.v)+'</details>';
    }).join('');
    inspector.innerHTML=back?backward(n,row,col):equation(forwardFormula[n.kind])+calc(n,row,col)+inputTables;
    inspector.querySelectorAll('details').forEach(el=>{if(open.has(el.firstElementChild.textContent)){el.open=true;}});
    inspector.querySelectorAll("[data-node]").forEach(b=>b.onclick=()=>select(+b.dataset.node));
    attachedWeights(n);
  }

  // Draw the selected value or gradient as a table with token and feature labels.
  function matrix(){
    const n=d.nodes[layer],values=back?n.g:n.v;
    row=Math.min(row,values.length-1);col=Math.min(col,values[0].length-1);
    $("title").textContent=n.name+" · "+values.length+" × "+values[0].length;
    $("why").textContent=(notes[n.kind]??"")+
      (n.output_start!==undefined?" This head uses its own weight rows and reads every input feature.":"");
    const isAtt=n.group!=="Prediction"&&["dot","mask","softmax"].includes(n.kind);
    const axis=isAtt?'input position':["Next-token scores","Probabilities"].includes(n.name)?'possible digit':n.name==="Cross-entropy"?'loss':'feature';
    $("axes").textContent=(n.name==="Cross-entropy"?'Rows = answer positions':'Rows = token positions')+' · columns = '+axis+'. Teal is positive, purple is negative. Stronger color means larger magnitude within this matrix.';
    const encoder=n.group.startsWith("Encoder");
    const rowTokens=n.kind==="loss"?d.target.map(x=>x===d.bos?d.eos:x):
      n.name.startsWith("Key")||n.name.startsWith("Value")?
        (encoder||n.group.includes("Cross-attention")?d.source:d.tokens):
        (encoder?d.source:d.tokens);
    const keyTokens=encoder||n.group.includes("Cross-attention")?d.source:d.tokens;
    const digitColumns=n.group==="Prediction"&&n.kind!=="loss";
    const columnLabel=i=>isAtt?i+' · '+token(keyTokens[i]):digitColumns?token(i===d.bos?d.eos:i):i;
    const max=Math.max(...values.flat().filter(v=>v>-1e29).map(Math.abs),1e-12);
    $("grid").style.gridTemplateColumns='max-content repeat('+values[0].length+', minmax(53px, 1fr))';
    const headings='<span></span>'+values[0].map((_,i)=>'<span class="idx">'+columnLabel(i)+'</span>').join('');
    $("grid").innerHTML=headings+values.map((v,r)=>'<span class="idx">'+r+' · '+token(rowTokens[r])+'</span>'+v.map((x,c)=>{
      const color=x<0?'142,6,142':'0,105,92', alpha=x< -1e29?0:Math.min(1,Math.abs(x)/max);
      const msg=n.name+' ['+r+', '+c+'] = '+f(x)+'. '+(notes[n.kind] ?? "");
      return '<button aria-label="'+msg+'" title="'+msg+'" data-r="'+r+'" data-c="'+c+'" aria-pressed="'+(row===r&&col===c)+'" style="background:rgba('+color+','+alpha+');color:'+(alpha>.6?'white':'#202124')+'">'+f(x)+'</button>';
    }).join('')).join('');
    $("grid").querySelectorAll("button").forEach(cell=>cell.onclick=()=>{
      $("grid").querySelector('[aria-pressed=true]').setAttribute("aria-pressed","false");
      row=+cell.dataset.r;col=+cell.dataset.c;parameterCell=null;cell.setAttribute("aria-pressed","true");math();
    });
    math();
  }
  function attachedWeights(n){
    const names=n.kind==="position"?["position"]:n.layer?
      [n.layer+".weight",n.layer+".bias"].filter(name=>name in matrixInitial):[];
    if(!names.length){$("weights").replaceChildren();return;}
    const open=new Set([...$("weights").querySelectorAll('details[open][data-tensor]')].map(el=>el.dataset.tensor));
    $("weights").innerHTML=names.map((name,i)=>{
      const wholeInitial=matrixInitial[name],wholeCurrent=d.params[name]?.before??wholeInitial;
      const offset=n.output_start??0;
      const limit=n.output_start===undefined?wholeInitial.length:offset+n.v[0].length;
      const initial=wholeInitial.slice(offset,limit),current=wholeCurrent.slice(offset,limit);
      const label=name==="position"?"Position table":name.endsWith(".weight")?"Weight matrix":"Bias vector";
      const purpose=name==="position"?"Fixed sine and cosine values":
        n.kind==="embedding"?"Rows are token IDs. PyTorch initializes the table with standard normal draws.":
        n.kind==="norm"?"One learned scale or shift per feature":
        n.output_start!==undefined?"These rows belong to this head. It reads every input feature.":
        name.endsWith(".bias")?"One bias per output feature":"Rows are output features. Columns are input features.";
      const isLinearWeight=n.kind==="linear"&&name.endsWith(".weight");
      const stored=matrixTable(current,n.kind==="embedding"?initial.map((_,k)=>token(k)):null,
        name==="position"?null:i,offset);
      const transpose=isLinearWeight?current[0].map((_,col)=>current.map(row=>row[col])):null;
      const matrices=isLinearWeight?
        '<div class="projection-weights"><section><b>Stored W · '+current.length+' × '+current[0].length+
          '</b>'+stored+'</section><section><b>Wᵀ used in XWᵀ · '+transpose.length+' × '+transpose[0].length+
          '</b>'+matrixTable(transpose)+'</section></div>':stored;
      return '<details data-tensor="' + escape(name) + '" '+(open.has(name)?'open':'')+'><summary>'+label+'</summary>'+
        '<p><code>'+escape(name)+'</code> · '+purpose+'</p><p>Before update '+d.update+
        '. '+(name==="position"?"Positions stay fixed.":"Click a value to see its gradient and Adam update.")+'</p>'+
        matrices+
        '<details><summary>Initial values · seed '+model.get("seed")+'</summary>'+
        matrixTable(initial,n.kind==="embedding"?initial.map((_,k)=>token(k)):null)+'</details></details>';
    }).join('')+'<div id="adam"></div>';
    $("weights").querySelectorAll('[data-weight]').forEach(b=>b.onclick=()=>{
      parameterCell={name:names[+b.dataset.weight],r:+b.dataset.wrow,c:+b.dataset.wcol};
      showAdam();
    });
    showAdam();
  }

  // Each operand links to its source tensor cell or learned parameter.
  function scalarGraph(n,r,c){
    const target=$("scalar");
    const [a,b]=n.parents.map(i=>d.nodes[i].v);
    const at=(parent,i,j,label)=>({label,value:d.nodes[parent].v[i][j],action:{node:parent,row:i,col:j}});
    const fromWeight=(name,i,j,label)=>{
      const values=d.params[name].before;
      return {label,value:Array.isArray(values[i])?values[i][j]:values[i],action:{name,r:i,c:j}};
    };
    const weight=n.layer+".weight",bias=n.layer+".bias";
    let pairs=[],operation;
    if(n.kind==="dot"){
      pairs=a[r].map((_,k)=>[at(n.parents[0],r,k,"Q["+r+","+k+"]"),at(n.parents[1],c,k,"K["+c+","+k+"]")]);
      operation="Sum ÷ √"+a[r].length;
    }else if(n.kind==="linear"){
      const weightRow=(n.output_start??0)+c;
      pairs=a[r].map((_,k)=>[at(n.parents[0],r,k,"x["+r+","+k+"]"),
        fromWeight(weight,weightRow,k,"W["+c+","+k+"]")]);
      operation="Sum + bias "+f(d.params[bias].before[weightRow]);
    }else if(n.kind==="weighted"){
      pairs=a[r].map((_,j)=>[at(n.parents[0],r,j,"A["+r+","+j+"]"),at(n.parents[1],j,c,"V["+j+","+c+"]")]);
      operation="Sum products";
    }else if(n.kind==="add"){
      pairs=[[at(n.parents[0],r,c,"Input A"),at(n.parents[1],r,c,"Input B")]];
      operation="Add";
    }else if(n.kind==="embedding"){
      pairs=[[fromWeight(weight,n.tokens[r],c,"Embedding weight"),{label:"√model width",value:Math.sqrt(d.width)}]];
      operation="Scale embedding";
    }
    if(!pairs.length){scalarSteps(n,r,c,at);return;}
    const nodes=[],edges=[];
    const rowSize=Math.min(pairs.length,3), stride=212, width=rowSize*stride;
    const transposeWeight=n.kind==="linear";
    const height=transposeWeight?152:104;
    const resultY=Math.ceil(pairs.length/rowSize)*height+20;
    pairs.forEach(([left,right],i)=>{
      const x=(i%rowSize)*stride+stride/2, y=Math.floor(i/rowSize)*height+23;
      const direct=n.kind==="add"||n.kind==="embedding";
      [left,right].forEach((operand,side)=>{
        const id=i+"-"+side;
        const operandX=x+(side?53:-53);
        nodes.push({id,action:operand.action,label:operand.label,sub:f(operand.value),x:operandX,y,w:100,h:36});
        if(transposeWeight&&side===1){
          const transposed="transpose-"+i;
          nodes.push({id:transposed,action:operand.action,label:"Transpose",sub:"Wᵀ["+i+","+c+"] = "+f(operand.value),
            tooltip:"Transpose swaps ["+c+","+i+"] to ["+i+","+c+"]. The value stays "+f(operand.value)+".",
            x:operandX,y:y+48,w:100,h:36});
          edges.push([id,transposed],[transposed,"product-"+i]);
        }else{
          edges.push([id,direct?"result":"product-"+i]);
        }
      });
      if(!direct){
        nodes.push({id:"product-"+i,label:"Multiply",sub:f(left.value*right.value),x,y:y+(transposeWeight?100:50),w:108,h:36});
        edges.push(["product-"+i,"result"]);
      }
    });
    nodes.push({id:"result",label:operation,sub:f(n.v[r][c]),selected:true,x:width/2,y:resultY,w:Math.min(width-10,190),h:36});
    drawGraph(target,nodes,edges,width,resultY+25,operand=>{
      if("node" in operand){select(operand.node,operand.row,operand.col);return;}
      parameterCell=operand;
      showAdam();
      $("adam").scrollIntoView({block:"nearest"});
    });
    target.querySelector("svg").style.minWidth=width+"px";
  }

  // Expand softmax into the row maximum, exponentials, sum, and selected division.
  function softmaxGraph(n,r,c,at){
    const input=n.parents[0], scores=d.nodes[input].v[r];
    const max=Math.max(...scores), exps=scores.map(score=>Math.exp(score-max));
    const stride=148, width=Math.max(300,stride*scores.length);
    const center=i=>(width-(scores.length-1)*stride)/2+i*stride;
    const nodes=[], edges=[];
    scores.forEach((score,i)=>{
      const x=center(i);
      nodes.push({id:"score-"+i,label:"Score "+i,sub:f(score),x,y:23,w:132,h:36,
        action:at(input,r,i,"Score "+i).action});
      nodes.push({id:"exp-"+i,label:"exp(score "+i+" − max)",sub:f(exps[i]),
        selected:i===c,x,y:132,w:132,h:36});
      edges.push(["score-"+i,"max"],["score-"+i,"exp-"+i],["max","exp-"+i],["exp-"+i,"sum"]);
    });
    nodes.push({id:"max",label:"Row maximum",sub:f(max),x:width/2,y:77,w:132,h:36});
    nodes.push({id:"selected",label:"Selected exponential",sub:f(exps[c]),x:width/4,y:187,w:140,h:36});
    nodes.push({id:"sum",label:"Sum of exponentials",sub:f(sum(exps)),x:3*width/4,y:187,w:140,h:36});
    nodes.push({id:"result",label:n.group==="Prediction"?"Softmax probability":"Softmax attention weight",
      sub:"Selected ÷ sum = "+f(n.v[r][c]),selected:true,x:width/2,y:246,w:186,h:36});
    edges.push(["exp-"+c,"selected"],["selected","result"],["sum","result"]);
    drawGraph($("scalar"),nodes,edges,width,272,operand=>select(operand.node,operand.row,operand.col));
    $("scalar").querySelector("svg").style.minWidth=Math.min(width,700)+"px";
  }

  // Reductions and elementwise operations also expose their scalar operands.
  function scalarSteps(n,r,c,at){
    const [input]=n.parents;
    const value=n.v[r][c];
    const term=(label,number)=>({label,value:number});
    let operands=[],operation,intermediate=[];
    if(n.kind==="softmax"){
      softmaxGraph(n,r,c,at);
      return;
    }
    if(n.kind==="norm"){
      const values=d.nodes[input].v[r], stats=normalize(values,n.eps);
      operands=values.map((x,j)=>at(input,r,j,"Feature "+j));
      intermediate=[term("Mean",stats.mean),term("Standard deviation",stats.sd),
        term("Normalized feature",stats.values[c])];
      operation="Scale and shift";
    }else if(n.kind==="relu"||n.kind==="mask"){
      operands=[at(input,r,c,n.kind==="relu"?"Input":"Unmasked score")];
      operation=n.kind==="relu"?"Max with zero":c<=r?"Keep score":"Replace with −∞";
    }else if(n.kind==="slice"){
      operands=[at(input,r,n.start+c,"Projected feature "+(n.start+c))];
      operation="Copy into head";
    }else if(n.kind==="join"){
      let offset=0;
      for(const parent of n.parents){
        const width=d.nodes[parent].v[r].length;
        if(c<offset+width){operands=[at(parent,r,c-offset,"Head feature "+(c-offset))];break;}
        offset+=width;
      }
      operation="Copy to joined vector";
    }else if(n.kind==="loss"){
      operands=[at(input,r,d.target[r],"Target probability")];
      operation="Negative natural log";
    }else if(n.kind==="position"){
      const power=2*Math.floor(c/2)/d.width;
      operands=[term("Position",r),term("Frequency",10000**(-power))];
      operation=c%2?"Cosine of product":"Sine of product";
    }
    if(!operation){$("scalar").replaceChildren();return;}
    const stride=136,width=Math.max(290,stride*Math.max(operands.length,intermediate.length));
    const layout=(list,stage,y)=>list.map((v,i)=>({id:stage+i,label:v.label,sub:f(v.value),action:v.action,
      x:(width-(list.length-1)*stride)/2+i*stride,y,w:128,h:36}));
    const inputs=layout(operands,"input",23),middle=layout(intermediate,"step",79);
    const resultY=middle.length?135:79;
    const result={id:"result",label:operation,sub:f(value),selected:true,x:width/2,y:resultY,w:190,h:36};
    const edges=middle.length?
      [...inputs.flatMap(v=>middle.map(t=>[v.id,t.id])),
        ...middle.map(v=>[v.id,result.id])]:inputs.map(v=>[v.id,result.id]);
    drawGraph($("scalar"),[...inputs,...middle,result],edges,width,resultY+24,
      operand=>select(operand.node,operand.row,operand.col));
    $("scalar").querySelector("svg").style.minWidth=Math.min(width,700)+"px";
  }

  // Show the gradient and optimizer change for the selected parameter.
  function showAdam(){
    const panel=$("adam");
    if(!panel){return;}
    if(!parameterCell){panel.replaceChildren();return;}
    const {name,r,c}=parameterCell, p=d.params[name];
    const matrix=Array.isArray(p.before[0]);
    const at=k=>matrix?p[k][r][c]:p[k][r];
    const [a,b]=d.betas;
    const m=at("m")/(1-a**d.update),v=at("v")/(1-b**d.update);
    const layerName=name.slice(0,name.lastIndexOf("."));
    const terms=d.nodes.filter(n=>n.layer===layerName).flatMap(n=>{
      if(n.kind==="embedding"){return n.g.map((g,t)=>n.tokens[t]===r?g[c]*Math.sqrt(d.width):0);}
      const x=d.nodes[n.parents[0]].v;
      if(n.kind==="norm"){return n.g.map((g,t)=>g[r]*(name.endsWith(".weight")?normalize(x[t],n.eps).values[r]:1));}
      const localRow=r-(n.output_start??0);
      if(localRow<0||localRow>=n.g[0].length){return [];}
      return n.g.map((g,t)=>g[localRow]*(matrix?x[t][c]:1));
    });
    const formula=name==="embedding.weight"?"g_E[t,c] = √d_model · Σ_{i : token[i]=t} g_x[i,c]":
      layerName.includes("_norm")?(name.endsWith(".weight")?"g_γ[c] = Σ_i g_y[i,c] · z[i,c]":"g_β[c] = Σ_i g_y[i,c]"):
      (matrix?"g_W[r,c] = Σ_i g_y[i,r] · x[i,c]":"g_b[r] = Σ_i g_y[i,r]");
    const initial=matrix?matrixInitial[name][r][c]:matrixInitial[name][r];
    panel.innerHTML='<h3>Adam update</h3><p><code>'+escape(name)+'['+r+(matrix?', '+c:'')+']</code></p>'+
      '<p>Initial '+f(initial)+' · before '+f(at("before"))+' · after <b>'+f(at("after"))+'</b></p>'+
      equation(formula)+'<p>This example contributes '+f(sum(terms))+' to the batch gradient '+f(at("g"))+'.</p>'+
      detail("Show this example’s contributions",fmt(terms))+
      equation("m = β₁·m_prev + (1−β₁)·g; v = β₂·v_prev + (1−β₂)·g²")+
      '<p>m = '+a+' × '+f(at("m0"))+' + '+f(1-a)+' × '+f(at("g"))+' = '+f(at("m"))+
      '<br>v = '+b+' × '+f(at("v0"))+' + '+f(1-b)+' × '+f(at("g"))+'² = '+f(at("v"))+'</p>'+
      equation("w_new = w − learning_rate · m̂ / (√v̂ + ε)")+
      '<p>Bias-corrected m̂ = '+f(m)+' and v̂ = '+f(v)+'. Subtract '+f(d.rate)+' × '+f(m)+' / (√'+f(v)+' + '+f(d.eps)+').</p>';
  }

  // Read the new replay packet, attach the loss nodes, and redraw the figures.
  function update(){
    const packet=model.get("packet"); if(!packet.nodes){return;}
    // Responses share one checkpoint across the inspector and generation panels.
    // Only initialize controls here so a reply cannot rewind a slider mid-drag.
    if(!d){
      $("step").value=packet.update;
      $("sample").value=packet.sample;
      $("custom").value=packet.source.join("")===packet.recorded.join("")?"":packet.source.join("");
      mark(packet.update);
    }
    d=structuredClone(packet);
    const logits=d.nodes.at(-1),p=logits.v.map(psoft);
    const pg=p.map((v,r)=>v.map((x,c)=>c===d.target[r]?-1/(x*d.count):0));
    d.nodes.push({name:"Probabilities",kind:"softmax",group:"Prediction",v:p,g:pg,parents:[d.nodes.length-1]});
    d.nodes.push({name:"Cross-entropy",kind:"loss",group:"Prediction",v:d.losses.map(x=>[x]),g:d.losses.map(()=>[1/d.count]),parents:[d.nodes.length-1]});
    $("status").textContent='Update '+d.update+' · example '+d.sample+' · inspecting '+d.source.join('')+' · '+(d.source.join('')===d.recorded.join('')?'recorded':'edited batch, simulated')+' loss '+f(d.loss)+'.';
    $("prefix").innerHTML='Before update '+d.update+' · Encoder reads <b>'+d.source.join(' ')+'</b><br>Decoder reads <b>'+d.tokens.map(token).join(' ')+'</b><br>Targets <b>'+d.target.map(x=>token(x===d.bos?d.eos:x)).join(' ')+'</b>';
    generation();repaint();
    if(inFlight?.id===packet.request){
      inFlight=null;
      sendReplay();
    }
  }
  // Register the captured Latin typefaces on the document for shadow-root text.
  for(const [family,source] of Object.entries(model.get("fonts"))){
    const font=new FontFace(family,'url('+source+')',{weight:'100 900'});
    document.fonts.add(font);font.load().catch(()=>{});
  }
  model.on("change:packet",update);update();
  return ()=>model.off("change:packet",update);
}
export default {render};
"""

In [ ]:
# Captured PAIR stylesheets retain their Apache 2.0 notices.
figure_css = '/* Copyright 2020 Google LLC. All Rights Reserved.\n\nLicensed under the Apache License, Version 2.0 (the "License");\nyou may not use this file except in compliance with the License.\nYou may obtain a copy of the License at\n\n    http://www.apache.org/licenses/LICENSE-2.0\n\nUnless required by applicable law or agreed to in writing, software\ndistributed under the License is distributed on an "AS IS" BASIS,\nWITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\nSee the License for the specific language governing permissions and\nlimitations under the License.\n==============================================================================*/\n\n\n\n\nhtml {\n  background-color: #fff;\n  font-weight: normal;\n  font-family: \'Roboto\', Helvetica, sans-serif;\n  font-weight: 300;\n  line-height: 1.55em;\n  font-size: 16px;\n}\n\n\nbody {\n  max-width: 850px;\n  margin: 0px auto;\n  margin-top: 5px;\n  margin-bottom: 80px;\n  color: #3C4043;\n  font-smoothing: antialiased;\n}\n\n@media (max-width: 760px) {\n  body {\n    padding: 5px;\n  }\n}\n\np {\n  line-height: 1.55em;\n  font-size: 16px;\n  /*line-height: 28px;*/\n  color: #3C4043;\n  letter-spacing: 0.1px;\n}\n\na {\n  color: black;\n}\n\n.headline {\n  margin-bottom: 40px;\n}\n\n.byline {\n  text-align: center;\n  font-size: 12px;\n  line-height: 1.3em;\n  margin-bottom: 40px;\n  margin-top: -20px;\n}\n\n.header {\n  position: relative;\n  color: black;\n  font-size: 16px;\n  height: 24px;\n  overflow: visible;\n  font-family: \'Google Sans\', sans-serif;\n  font-weight: 100;\n  font-size: 20px;\n  margin: 0px auto;\n  margin-top: 15px;\n  padding-left: 20px;\n}\n\n.header-left {\n  vertical-align: middle;\n  font-size: 20px;\n  margin: 0px auto;\n  width: 300px;\n}\n\n.header-left img {\n  width: 100px;\n  opacity: 1;\n  top: 0px;\n  position: relative;\n}\n\n.header-left a:first-child {\n  float: left;\n}\n\n.header-left a:last-child {\n  position: relative;\n  top: 8px;\n  margin-left: 20px;\n  float: left;\n}\n\n.header-left a {\n  line-height: 20px;\n  -webkit-font-smoothing: antialiased;\n  letter-spacing: 0.1px;\n  font-size: 20px;\n  text-transform: uppercase;\n  font-family: "Google Sans";\n  text-align: right;\n  -webkit-tap-highlight-color: rgba(255, 255, 255, 0);\n  font-weight: 300;\n  text-decoration: none;\n  /*margin: 50px 0 0 50px;*/\n  display: inline-block;\n  color: #00695C !important;\n}\n\n.header-left a:hover {\n  color: #ff4081 !important;\n}\n\n@media (max-width: 750px) {\n  .header-right span {\n    opacity: 0;\n  }\n}\n\n.header a {\n  /*opacity: .5;*/\n  text-decoration: none;\n}\n\n.header a:hover {\n  opacity: 1\n}\n\n\np {\n  max-width: 750px;\n  margin: 0px auto;\n  margin-block-start: 1em;\n  margin-block-end: 1em;\n}\n\n/*TODO mobile padding?*/\n\nh3 {\n  max-width: 750px;\n  margin: 0px auto;\n  font-weight: 100;\n  line-height: 1.3em;\n}\n\nh1,\nh2,\nh3,\nh4,\nh5 {\n  font-family: \'Google Sans\', sans-serif;\n  font-weight: 100;\n  margin-top: 1.5em;\n  margin-bottom: .5em;\n}\n\nh1 {\n  font-weight: 100;\n  font-size: 34px;\n  margin-bottom: .5em;\n  line-height: 1.3em;\n  margin-top: 1.4em;\n  text-align: center;\n  font-family: "Google Sans";\n  /*color: #00695C;*/\n}\n\nh2,\nh3,\nh4,\nh5 {\n  font-size: 22px;\n}\n\n/*wp classes*/\nimg.aligncenter {\n  display: block;\n  margin: auto;\n  max-width: 750px;\n}\n\n\n\nhtml {\n  overflow-x: hidden;\n}\n\n.full-width {\n  width: 100vw;\n  position: relative;\n  left: 50%;\n  right: 50%;\n  margin-left: -50vw;\n  margin-right: -50vw;\n  overflow: hidden;\n}\n\n.full-width img {\n  max-width: 100%;\n  display: block;\n  margin: 0 auto;\n}\n\n.full-width.px980 img,\n.full-width.px980 div {\n  max-width: 980px;\n}\n\n.full-width>div,\n.full-width>div>div {\n  margin: 0px auto;\n}\n\n.full-width.px750 img,\n.full-width.px750 div {\n  max-width: 750px;\n}\n\ndraft {\n  display: none;\n  /*visibility: collapse;*/\n}\n\n\nh1,\n.post-summary {\n  max-width: 750px;\n  margin: 0px auto;\n}\n\n.post-summary {\n  font-size: 19px;\n  margin-bottom: 65px;\n  line-height: 1.5em;\n}\n\nh1 {\n  margin-bottom: 40px;\n  margin-top: 50px;\n}\n\n.post-tags {\n  line-height: 1.55em;\n  font-style: italic;\n}\n\n.thumbnail-caption {\n  font-style: italic;\n}\n\n\n\n\n\n\n/*graph scroll stuff*/\n\n#container {\n  position: relative;\n  width: 900px;\n  margin-left: -25px;\n}\n\n#container h3 {\n  line-height: 1.3em;\n}\n\n\n\n\n\n\n.tooltip {\n  top: -1000px;\n  position: fixed;\n  padding: 10px;\n  background: rgba(255, 255, 255, .90);\n  border: 1px solid lightgray;\n  pointer-events: none;\n  width: 300px;\n}\n\n.tooltip-hidden {\n  opacity: 0;\n  transition: all .3s;\n  transition-delay: .1s;\n}\n\n@media (max-width: 590px) {\n  div.tooltip {\n    bottom: -1px;\n    width: calc(100%);\n    left: -1px !important;\n    right: -1px !important;\n    top: auto !important;\n    width: auto !important;\n  }\n}\n\n\n\n\n.footend {\n  margin-left: -9px;\n  width: 10px;\n}\n\n\n.footstart,\n.footend {\n  text-decoration: none;\n}\n\n.footstart:hover,\n.footend:hover {\n  text-decoration: underline;\n}\n\n\n\n\n#recirc {}\n\n#recirc .img {\n  outline: 1px solid #ccc;\n}\n\n#recirc .post:hover .img {\n  outline: 1px solid #333;\n}\n\n#recirc .title {\n  /*color: #00695C;*/\n  font-size: 18px;\n  font-weight: 500;\n  margin-bottom: -10px;\n  /*height: 10px !important;*/\n  /*opacity: 0;*/\n}\n\n#recirc .post:hover .title {\n  text-decoration: underline !important;\n}\n\n#recirc .post {\n  margin-bottom: 30px;\n}\n\n\n\n\n\n\n\n\n\n\n\n\n\n/*Nav Style*/\n#nav-container {\n  width: 100vw;\n  margin-left: calc(50% - 50vw);\n  display: inline-block;\n  /*display: none;*/\n}\n\n#navigation {\n  margin: 0 auto;\n  max-width: 1260px;\n  -webkit-font-smoothing: antialiased;\n  font-family: \'Open Sans\', Helvetica, sans-serif;\n  font-weight: 300;\n  letter-spacing: 0.1px;\n\n\n  color: rgba(0, 0, 0, .87);\n  font-size: 14px;\n  line-height: 20px;\n  -webkit-font-smoothing: antialiased;\n  font-family: \'Open Sans\', Helvetica, sans-serif;\n  font-weight: 300;\n  letter-spacing: 0.1px;\n  display: flex;\n  flex-flow: row wrap;\n  align-items: stretch;\n  padding: 8px;\n  margin: 0 auto;\n  max-width: 1260px;\n}\n\n.mdl-grid {\n  display: -webkit-flex;\n  display: -ms-flexbox;\n  display: flex;\n  -webkit-flex-flow: row wrap;\n  -ms-flex-flow: row wrap;\n  flex-flow: row wrap;\n  margin: 0 auto;\n  -webkit-align-items: stretch;\n  -ms-flex-align: stretch;\n  align-items: stretch;\n}\n\n.mdl-cell {\n  box-sizing: border-box;\n}\n\n.nav-links {\n  font-size: 20px;\n  text-transform: uppercase;\n  font-family: "Google Sans";\n  color: #4a4a4a;\n  text-align: right;\n}\n\n.nav-logo-small {\n  width: 110px;\n  margin: 42px 0 0 0;\n}\n\n.nav-links .selected {\n  color: #00695C !important;\n}\n\n/*.nav-links a:visited {\n  color: #4a4a4a;\n}\na:visited {\n  color: #7B1FA2;\n}\n*/\n.nav-links a {\n  color: inherit;\n  text-decoration: none;\n  margin: 50px 0 0 50px;\n  display: inline-block;\n}\n\n\n@media screen and (max-width: 1035px) {\n  .nav-links {\n    font-size: 16px;\n  }\n}\n\n.nav-links {\n  line-height: 20px;\n  -webkit-font-smoothing: antialiased;\n  font-weight: 300;\n  letter-spacing: 0.1px;\n  box-sizing: border-box;\n  margin: 8px;\n  width: calc(66.6666666667% - 16px);\n  font-size: 20px;\n  text-transform: uppercase;\n  font-family: "Google Sans";\n  color: #4a4a4a;\n  text-align: right;\n}\nbody{\n\n}\n\n\np{\n  margin-left: 0px auto;\n  margin-right: 0px auto;\n  margin: 0px auto;\n  margin-top: 1em;\n  margin-bottom: 1em;\n}\nh3, .post-summary, h1x, p{\n  max-width: 650px;\n}\n\n#recirc{\n  max-width: 760px;\n}\n\n\n.white{\n  stroke: #fff;\n  fill: none;\n  stroke-width: 1;\n}\n\n.player{\n\tcursor: pointer;\n  stroke: #000;\n  stroke-width: 2;\n}\n\n.button{\n  border: .5px solid #000;\n  /*border-bottom-width: 4px;*/\n  /*border-right-width: 4px;*/\n  border-radius: 8px;\n  padding: 4px;\n  margin: 2px;\n  cursor: pointer;\n  display: inline-block;\n  /*font-family: monospace;*/\n  /*font-family: \'Roboto Slab\', serif;*/\n  /*font-size: 16px;*/\n  user-select: none;\n  font-family: \'Google Sans\', sans-serif;\n  font-family: \'Roboto\', Helvetica, sans-serif;\n\n  /*font-weight: 300;*/\n}\n\n@media (min-width: 800px){\n  .button{\n    margin-bottom: -100px;\n  }\n}\n\n.inline-button{\n  display: inline;\n}\n\n.button:hover{\n  background: #eee !important;\n}\n\n.button:active{\n}\n\ncanvas{\n  opacity: .9;\n}\n\nsvg{\n  overflow: visible;\n}\n\n.axis{\n  font-size: 12px;\n\n}\n.axis{\n  color: #000;\n}\n.axis text{\n  fill: #999;\n  font-family: \'Roboto\', Helvetica, sans-serif;\n}\n.axis text.chart-title{\n  fill: #000;\n  font-size: 16px;\n}\n.axis line{\n  stroke: #ccc;\n  display: none;\n}\n\n.domain{\n  stroke: #ccc;\n  display: none;\n}\n\ntext, .chart-title{\n  user-select: none;\n  /*pointer-events: none;*/\n}\n\n\n.field{\n  font-family: \'Google Sans\', sans-serif;\n  font-family: \'Roboto\', Helvetica, sans-serif;\n  margin-top: 10px;\n}\n\n.chart-title span{\n  padding: 4px;\n}\n\n.chart-title span:last-child{\n  color: #fff;\n}\n\n.chart-title span:first-child{\n  color: #000;\n}\n\n#field-regression .white, #field-regression-leak .white{\n\tstroke: #ccc;\n}\n\n#field-grass .button, #field-prediction .button{\n  display: none;\n}\n\n.face-container{\n  max-width: 400px;\n\n  margin: 0px auto;\n}\n.face-container img{\n  width: 100%;\n}\n\n.post-summary {\n  margin-bottom: 40px;\n}\n\np {\n  margin: 10 auto;\n}\n\n\n\n.pointer{\n  height: 0px;\n  position: relative;\n}\n.pointer div {\n  overflow: visible;\n  content: "";\n  background-image: url(https://pair-code.github.io/interpretability/bert-tree/pointer.svg);\n  width: 27px;\n  height: 27px;\n  position: absolute;\n  left: -35px;\n  top: 0px;\n}\n\n\n.face-container:after{\n  content: "M. Fredrikson, S. Jha, and T. Ristenpart, “Model inversion attacks that exploit confidence information and basic countermeasures,” in CCS, 2015.";\n  font-size: 12px;\n  color: #888;\n  line-height: 14px;\n  display: block;\n}\n:host{display:block;background:white}\nmain{font:300 16px/1.55 Roboto,Helvetica,sans-serif;color:#3c4043;max-width:850px;margin:auto;padding:0 10px 36px}\nmain p,main h3,.controls,#sequence,#odds,#adam,#curve,#prefix{max-width:650px;margin-left:auto;margin-right:auto}\nmain h1,main h3{font-family:"Google Sans",sans-serif;font-weight:400;line-height:1.3}\nmain h1{text-align:center;font-size:34px;margin:50px 0 40px}main h3{font-size:22px;margin-top:1.5em}\nmain .post-summary{font-size:19px;margin-bottom:40px}main svg{display:block;width:100%}svg text{font:12px Roboto,Helvetica,sans-serif;fill:#3c4043}\n.controls{display:flex;gap:8px;flex-wrap:wrap;align-items:center}label{display:inline-flex;gap:4px;align-items:center}button,select,input{font:inherit;color:inherit}button,select{border:1px solid #999;background:white;border-radius:8px;padding:3px 7px;cursor:pointer}button:hover,button:focus-visible{background:#eee}button[aria-pressed=true]{outline:2px solid #00695c}input[type=number]{width:4em}input[type=range]{accent-color:#00695c}\n#odds{padding-top:10px}#odds div{display:grid;grid-template-columns:18px 1fr 84px;align-items:center;gap:5px;font-size:12px;height:17px}#odds i{height:8px;background:#00695c}#odds small{font-size:12px;text-align:right}\n.work{display:grid;grid-template-columns:minmax(210px, .75fr) minmax(0,1fr);align-items:start;gap:16px;margin:20px auto}.work section{position:sticky;top:12px;min-width:0}.work h3{margin:0}.work p{margin:.6em 0}#graph g{cursor:pointer}#graph g:hover rect,#graph g:focus rect{stroke:#000;stroke-width:2}\n#axes{font-size:13px}#grid{display:grid;overflow:auto;gap:1px;padding:3px}#grid button{height:17px;min-width:10px;padding:0;border:1px solid #ccc;border-radius:0}#grid button[aria-pressed=true]{outline:2px solid black;z-index:1}\ndetails{margin:.5em 0}summary{cursor:pointer;text-decoration:underline}.numbers{max-height:220px;overflow:auto;overflow-wrap:anywhere}\n@media(max-width:680px){.work{grid-template-columns:1fr}.work section{position:static}#graph{max-height:420px}main h1{font-size:30px}}\n\n#graph .streams{display:grid;grid-template-columns:1fr 1fr;gap:8px}\n#graph button{width:100%;min-height:32px;font-size:13px}\n#graph button[aria-pressed=true]{background:#fde100}\n#graph .down{display:block;text-align:center;color:#00695c;line-height:1.2}\n#graph .link-note{font-size:13px;color:#00695c;text-align:center}\n#odds div{grid-template-columns:36px 1fr 84px}\n#odds div{display:flex} #odds b{width:36px;flex:none} #odds small{margin-left:auto;flex:none}\n#odds i{max-width:calc(100% - 120px)}\n#operation{max-width:100%;min-width:0}\n#grid .idx{white-space:nowrap;font-size:10px;padding:0 2px;text-align:center}\n'
figure_fonts = {'Roboto': 'https://fonts.gstatic.com/s/roboto/v51/KFO7CnqEu92Fr1ME7kSn66aGLdTylUAMa3yUBHMdazQ.woff2', 'Google Sans': 'https://fonts.gstatic.com/s/googlesans/v76/4UasrENHsxJlGDuGo1OIlJfC6l_24rlCK1Yo_Iqcsih3SAyH6cAwhX9RPjIUvbQoi-E.woff2'}

In [ ]:
import anywidget
import traitlets

# Widgets sync these fields between Python and the diagram in the browser.
class Explorer(anywidget.AnyWidget):
    _esm = diagram_js
    css = traitlets.Unicode(figure_css).tag(sync=True)
    fonts = traitlets.Dict(figure_fonts).tag(sync=True)
    packet = traitlets.Dict().tag(sync=True)
    selection = traitlets.Dict().tag(sync=True)
    history = traitlets.List().tag(sync=True)
    batch = traitlets.Int(BATCH).tag(sync=True)
    length = traitlets.Int(LENGTH).tag(sync=True)
    seed = traitlets.Int(SEED).tag(sync=True)
    initial_tensors = traitlets.Dict().tag(sync=True)

    # When the browser changes the selection, send back a fresh replay.
    @traitlets.observe("selection")
    def _on_selection(self, change):
        selection = change.new
        result = replay(selection["update"], selection["sample"] - 1, selection["digits"])
        self.packet = dict(result, request=selection["id"])

## Explore the run

Open **Decoder → Cross-attention**, then choose **Scores · head 1**. Click row 0, column 1 to see how the BOS query scores the second source digit. Click a query or key value to follow it upstream. Switch to **Backward** to see gradients for the same cell. The slider keeps the selected calculation in view as the weights change.

In [ ]:
viewer = Explorer(
    initial_tensors={name: values.tolist() for name, values in history[0][0].items()},
    history=[s[-1] for s in history],
    packet=replay(1, 0, "".join(map(str, HOLDOUT))),
)
display(viewer)